# HW02: Medical Image Segmentation and NLP

**Course**: CSYE 7374 - Deep Learning and Generative AI in Healthcare

---

## Objectives

In this homework, you will implement attention-based deep learning models for two core healthcare tasks:

1. **3D Medical Image Segmentation** — Build an Attention U-Net for volumetric medical image data
2. **Character-Level RNN** — Train an LSTM language model on medical text
3. **RNN with Attention for Text Classification** — Classify clinical notes using an attention-equipped RNN

---

## Instructions

- Complete all cells marked with **`# TODO`**
- Do not modify the provided helper functions or given code unless instructed
- Run all cells in order before submitting
- Answer the analysis questions at the end in the provided markdown cells

---

## Grading Rubric

| Task | Points |
|------|--------|
| P1: Attention gate implementation | 15 |
| P1: Attention U-Net up-block | 10 |
| P1: Combined Dice + CE loss | 10 |
| P1: Segmentation training loop | 10 |
| P2: Character-level LSTM model class | 10 |
| P2: RNN training loop | 5 |
| P2: Text generation function | 5 |
| P3: Attention mechanism implementation | 10 |
| P3: RNN-with-attention model class | 5 |
| P3: Classification training loop + evaluation | 5 |
| P3: Attention weight visualization | 5 |
| Analysis questions (4 × 5 points) | 20 |
| **Total** | **100** |

---
## 1. Setup and Imports

In [ ]:
# Install required packages (run once)
!pip install -q torch torchvision medmnist matplotlib seaborn scikit-learn tqdm pandas numpy

In [ ]:
import os
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# MedMNIST
import medmnist
from medmnist import INFO

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Metrics
from sklearn.metrics import (
    classification_report, accuracy_score,
    precision_recall_fscore_support, confusion_matrix
)

# Progress bar
from tqdm.notebook import tqdm

plt.style.use('seaborn-v0_8-whitegrid')
print(f"PyTorch: {torch.__version__}")
print(f"MedMNIST: {medmnist.__version__}")

---
## 2. Configuration

In [ ]:
class Config:
    # ── Problem 1: Segmentation ──────────────────────────────
    SEG_DATA_FLAG   = 'nodulemnist3d'   # MedMNIST 3D dataset
    SEG_BATCH_SIZE  = 4
    SEG_EPOCHS      = 10
    SEG_LR          = 1e-3
    SEG_IMG_SIZE    = 28               # NoduleMNIST3D native resolution
    SEG_FEATURES    = [16, 32, 64]     # Encoder channel widths

    # ── Problem 2: Character-level RNN ───────────────────────
    RNN_SEQ_LEN     = 100
    RNN_BATCH_SIZE  = 64
    RNN_EPOCHS      = 20
    RNN_LR          = 2e-3
    RNN_HIDDEN_SIZE = 256
    RNN_NUM_LAYERS  = 2
    RNN_DROPOUT     = 0.3

    # ── Problem 3: RNN + Attention Classification ──────────────
    CLS_BATCH_SIZE  = 32
    CLS_EPOCHS      = 15
    CLS_LR          = 1e-3
    CLS_HIDDEN_SIZE = 128
    CLS_EMBED_DIM   = 64
    CLS_MAX_LEN     = 50               # Max tokens per sample

    # ── General ─────────────────────────────────────────
    SEED            = 42
    CHECKPOINT_DIR  = './checkpoints_hw02'

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(Config.SEED)
os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## Problem 1: 3D Medical Image Segmentation with Attention U-Net

In this problem you will build an **Attention U-Net** for volumetric medical image segmentation.  
Attention gates learn to suppress irrelevant feature activations and amplify salient ones, making them particularly effective for small-structure segmentation.

**Dataset**: [NoduleMNIST3D](https://medmnist.com/) — lung nodule CT volumes, binary segmentation task  
(28×28×28 voxels per volume, downloaded automatically via `medmnist`)

---
### 1a. Data Loading

In [ ]:
# ── Given: Data loading for NoduleMNIST3D ───────────────────────────────────────────
import medmnist
from medmnist import NoduleMNIST3D
import torchvision.transforms as T

def load_seg_datasets():
    """Load NoduleMNIST3D train/val/test splits."""
    info = INFO['nodulemnist3d']
    print(f"Dataset: NoduleMNIST3D")
    print(f"  Task : {info['task']}")
    print(f"  n_channels: {info['n_channels']}")
    print(f"  n_classes : {len(info['label'])}")

    train_ds = NoduleMNIST3D(split='train', download=True)
    val_ds   = NoduleMNIST3D(split='val',   download=True)
    test_ds  = NoduleMNIST3D(split='test',  download=True)
    return train_ds, val_ds, test_ds

train_seg_raw, val_seg_raw, test_seg_raw = load_seg_datasets()
print(f"\nSplit sizes — Train: {len(train_seg_raw)} | Val: {len(val_seg_raw)} | Test: {len(test_seg_raw)}")

In [ ]:
# ── Given: Dataset wrapper that converts to tensors ──────────────────────────────
class Nodule3DSegDataset(Dataset):
    """
    Wraps NoduleMNIST3D so each sample returns:
      image  : FloatTensor [1, D, H, W]  (voxel intensities, normalised 0-1)
      target : FloatTensor [1, D, H, W]  (binary segmentation mask)
    """
    def __init__(self, base_dataset):
        self.data = base_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img, label = self.data[idx]
        # img is a numpy array [D, H, W] or [1, D, H, W]; normalise to [0, 1]
        img = np.array(img, dtype=np.float32)
        if img.ndim == 3:
            img = img[np.newaxis]          # add channel dim → [1, D, H, W]
        img = (img - img.min()) / (img.max() - img.min() + 1e-6)

        # For NoduleMNIST3D the label scalar indicates nodule presence;
        # we create a simple binary "has-nodule" mask by thresholding the image.
        # (A real pipeline would use voxel-level GT masks.)
        mask = (img > 0.5).astype(np.float32)   # shape [1, D, H, W]

        return torch.tensor(img), torch.tensor(mask)

train_seg_ds = Nodule3DSegDataset(train_seg_raw)
val_seg_ds   = Nodule3DSegDataset(val_seg_raw)
test_seg_ds  = Nodule3DSegDataset(test_seg_raw)

train_seg_loader = DataLoader(train_seg_ds, batch_size=Config.SEG_BATCH_SIZE, shuffle=True,  num_workers=0)
val_seg_loader   = DataLoader(val_seg_ds,   batch_size=Config.SEG_BATCH_SIZE, shuffle=False, num_workers=0)
test_seg_loader  = DataLoader(test_seg_ds,  batch_size=Config.SEG_BATCH_SIZE, shuffle=False, num_workers=0)

# Sanity check
sample_img, sample_mask = train_seg_ds[0]
print(f"Image shape : {sample_img.shape}   dtype: {sample_img.dtype}")
print(f"Mask  shape : {sample_mask.shape}  dtype: {sample_mask.dtype}")

In [ ]:
# ── Given: Visualise a mid-axial slice ────────────────────────────────────────────
def visualise_volume(image, mask, title='Sample Volume'):
    """Display centre axial slice of image and mask side-by-side."""
    d = image.shape[1]
    mid = d // 2
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(image[0, mid], cmap='gray')
    axes[0].set_title('Image (centre slice)')
    axes[0].axis('off')
    axes[1].imshow(mask[0, mid], cmap='hot')
    axes[1].set_title('Mask (centre slice)')
    axes[1].axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

visualise_volume(sample_img.numpy(), sample_mask.numpy())

---
### 1b. Model — Attention U-Net

The Attention U-Net adds **attention gates** to the skip connections of a standard U-Net.  
Each gate takes the skip-connection feature map and a gating signal from the decoder, computes  
soft attention coefficients, and re-weights the skip features before concatenation.

You need to implement:
1. `AttentionGate` — the core attention mechanism  
2. `UpBlockWithAttention` — up-sampling + attention + conv block  
3. The `AttentionUNet3D` forward pass uses the given encoder and bottleneck.

In [ ]:
# ── Given: Encoder building blocks ───────────────────────────────────────────────
class ConvBlock3D(nn.Module):
    """Two consecutive Conv3d → BN → ReLU layers."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class EncoderBlock3D(nn.Module):
    """ConvBlock followed by MaxPool3d. Returns (pooled, skip)."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv  = ConvBlock3D(in_ch, out_ch)
        self.pool  = nn.MaxPool3d(kernel_size=2, stride=2)

    def forward(self, x):
        skip   = self.conv(x)
        pooled = self.pool(skip)
        return pooled, skip

In [ ]:
class AttentionGate3D(nn.Module):
    """
    Attention gate for 3-D feature maps.

    Given:
      g  : gating signal from decoder      shape [B, g_ch, D, H, W]
      x  : skip connection from encoder    shape [B, x_ch, D, H, W]

    Returns:
      x * alpha : attended skip features   shape [B, x_ch, D, H, W]

    Architecture:
      1. Project g → inter_ch with 1×1×1 conv (W_g)
      2. Project x → inter_ch with 1×1×1 conv (W_x)
      3. Add projections, apply ReLU
      4. Project sum → 1 channel with 1×1×1 conv (psi), apply Sigmoid
      5. Upsample alpha to match x spatial size, multiply with x
    """
    def __init__(self, g_ch, x_ch, inter_ch):
        super().__init__()
        # ============================================================
        # TODO: Define W_g, W_x, psi as nn.Sequential blocks
        #       Each should be: Conv3d → BatchNorm3d
        #       psi should end with: Conv3d(inter_ch, 1) → BatchNorm3d
        # ============================================================
        self.W_g   = None  # TODO
        self.W_x   = None  # TODO
        self.psi   = None  # TODO
        self.relu  = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()
        # ============================================================

    def forward(self, g, x):
        # ============================================================
        # TODO: Implement the forward pass described in the docstring
        # ============================================================
        pass  # Remove after completing TODO
        # ============================================================

In [ ]:
class UpBlockWithAttention3D(nn.Module):
    """
    Decoder up-block with attention gate on the skip connection.

    Steps:
      1. Upsample decoder feature map x by 2× (use nn.Upsample, trilinear)
      2. Apply AttentionGate3D to get attended skip features
      3. Concatenate upsampled x with attended skip along channel dim
      4. Apply ConvBlock3D to fuse features
    """
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        inter_ch = max(out_ch // 2, 1)
        # ============================================================
        # TODO: Define self.upsample, self.attention, self.conv
        # ============================================================
        self.upsample  = None  # TODO: nn.Upsample(scale_factor=2, ...)
        self.attention = None  # TODO: AttentionGate3D(...)
        self.conv      = None  # TODO: ConvBlock3D(in_ch + skip_ch, out_ch)
        # ============================================================

    def forward(self, x, skip):
        # ============================================================
        # TODO: Implement the forward pass described in the docstring
        # ============================================================
        pass  # Remove after completing TODO
        # ============================================================

In [ ]:
# ── Given: Full Attention U-Net using your blocks ──────────────────────────────────
class AttentionUNet3D(nn.Module):
    """
    3-D Attention U-Net.
    Input : [B, 1, 28, 28, 28]
    Output: [B, 1, 28, 28, 28]  (logits for binary segmentation)
    """
    def __init__(self, in_ch=1, features=None):
        super().__init__()
        if features is None:
            features = Config.SEG_FEATURES     # e.g. [16, 32, 64]

        f = features
        # Encoder
        self.enc1 = EncoderBlock3D(in_ch,  f[0])
        self.enc2 = EncoderBlock3D(f[0],   f[1])
        # Bottleneck
        self.bottleneck = ConvBlock3D(f[1], f[2])
        # Decoder (your implementation)
        self.dec2 = UpBlockWithAttention3D(f[2], f[1], f[1])
        self.dec1 = UpBlockWithAttention3D(f[1], f[0], f[0])
        # Final 1×1×1 projection
        self.final = nn.Conv3d(f[0], 1, kernel_size=1)

    def forward(self, x):
        # Encoder
        x1, skip1 = self.enc1(x)    # skip1: [B, f[0], 28, 28, 28]
        x2, skip2 = self.enc2(x1)   # skip2: [B, f[1], 14, 14, 14]
        # Bottleneck
        x3 = self.bottleneck(x2)    # [B, f[2],  7,  7,  7]
        # Decoder
        x4 = self.dec2(x3, skip2)   # [B, f[1], 14, 14, 14]
        x5 = self.dec1(x4, skip1)   # [B, f[0], 28, 28, 28]
        return self.final(x5)       # [B,   1,  28, 28, 28]

# Instantiate and verify
seg_model = AttentionUNet3D().to(device)
dummy = torch.zeros(2, 1, 28, 28, 28).to(device)
out   = seg_model(dummy)
print(f"Model output shape: {out.shape}")   # Expected: [2, 1, 28, 28, 28]
total_params = sum(p.numel() for p in seg_model.parameters())
print(f"Total parameters  : {total_params:,}")

---
### 1c. Loss Function — Combined Dice + Cross-Entropy

In [ ]:
def dice_score(pred_logits, target, threshold=0.5):
    """
    Compute Dice score between predicted logits and binary target.
    pred_logits: [B, 1, D, H, W] raw logits
    target      : [B, 1, D, H, W] binary float mask
    """
    pred = (torch.sigmoid(pred_logits) > threshold).float()
    pred_flat   = pred.view(-1)
    target_flat = target.view(-1)
    intersection = (pred_flat * target_flat).sum()
    return (2.0 * intersection + 1e-6) / (pred_flat.sum() + target_flat.sum() + 1e-6)


class CombinedSegLoss(nn.Module):
    """
    Combined Dice Loss + Binary Cross-Entropy Loss.
    Final loss = dice_loss + bce_loss
    where dice_loss = 1 - dice_coefficient (computed on sigmoid probabilities).
    """
    def __init__(self):
        super().__init__()
        # ============================================================
        # TODO: Define self.bce as nn.BCEWithLogitsLoss()
        # ============================================================
        self.bce = None  # TODO
        # ============================================================

    def forward(self, pred_logits, target):
        """
        pred_logits: [B, 1, D, H, W]
        target      : [B, 1, D, H, W]
        Returns scalar combined loss.
        """
        # ============================================================
        # TODO: Compute BCE loss using self.bce
        # TODO: Compute Dice coefficient on sigmoid(pred_logits) and target
        #       (use soft dice: work on probabilities, not thresholded predictions)
        # TODO: Return bce_loss + (1 - dice_coeff)
        # ============================================================
        pass  # Remove after completing TODO
        # ============================================================

seg_criterion = CombinedSegLoss().to(device)
seg_optimizer = optim.Adam(seg_model.parameters(), lr=Config.SEG_LR)
print("Segmentation loss and optimizer ready.")

---
### 1d. Training Loop

In [ ]:
def train_seg_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_dice, n = 0.0, 0.0, 0

    for images, masks in tqdm(loader, desc='Seg Train', leave=False):
        images = images.to(device)
        masks  = masks.to(device)

        # ============================================================
        # TODO: Forward pass, compute loss, backward pass, optimizer step
        # ============================================================
        optimizer.zero_grad()
        preds = None    # TODO: forward pass
        loss  = None    # TODO: compute criterion
        # TODO: loss.backward() and optimizer.step()
        # ============================================================

        total_loss += loss.item() * images.size(0)
        total_dice += dice_score(preds.detach(), masks).item() * images.size(0)
        n          += images.size(0)

    return total_loss / n, total_dice / n


def eval_seg_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, total_dice, n = 0.0, 0.0, 0

    with torch.no_grad():
        for images, masks in tqdm(loader, desc='Seg Eval', leave=False):
            images = images.to(device)
            masks  = masks.to(device)
            preds  = model(images)
            loss   = criterion(preds, masks)
            total_loss += loss.item() * images.size(0)
            total_dice += dice_score(preds, masks).item() * images.size(0)
            n          += images.size(0)

    return total_loss / n, total_dice / n


def train_segmentation(model, train_loader, val_loader, criterion, optimizer, epochs, device):
    history = {'train_loss': [], 'val_loss': [], 'train_dice': [], 'val_dice': []}
    best_dice, best_weights = 0.0, None

    print(f"\n{'='*50}\nTraining Attention U-Net\n{'='*50}")
    for epoch in range(epochs):
        # ============================================================
        # TODO: Call train_seg_epoch and eval_seg_epoch,
        #       append results to history, print epoch summary,
        #       and save best model weights by val_dice
        # ============================================================
        pass  # Remove after completing TODO
        # ============================================================

    if best_weights:
        model.load_state_dict(best_weights)
    return model, history

In [ ]:
# Run segmentation training
seg_model, seg_history = train_segmentation(
    seg_model, train_seg_loader, val_seg_loader,
    seg_criterion, seg_optimizer,
    Config.SEG_EPOCHS, device
)

In [ ]:
# ── Given: Plot segmentation training curves ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(seg_history['train_loss'], label='Train Loss')
axes[0].plot(seg_history['val_loss'],   label='Val Loss')
axes[0].set_title('Segmentation Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(seg_history['train_dice'], label='Train Dice')
axes[1].plot(seg_history['val_dice'],   label='Val Dice')
axes[1].set_title('Dice Score')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

# Final test evaluation
test_loss, test_dice = eval_seg_epoch(seg_model, test_seg_loader, seg_criterion, device)
print(f"\nTest Loss: {test_loss:.4f} | Test Dice: {test_dice:.4f}")

---
## Problem 2: Character-Level RNN for Medical Text Generation

In this problem you will train a **character-level LSTM** language model on a medical text corpus.  
Given a seed string, the trained model will generate new text that mimics the style and vocabulary  
of medical clinical notes.

In [ ]:
# ── Given: Medical text corpus and character vocabulary ────────────────────────────────
MEDICAL_TEXT = """
The patient presents with acute chest pain radiating to the left arm, diaphoresis, and shortness of breath.
Blood pressure is 145 over 92 mmHg. Heart rate is 102 beats per minute. Oxygen saturation is 94 percent.
Electrocardiogram shows ST-segment elevation in leads II, III, and aVF consistent with inferior wall myocardial infarction.
Troponin I level is elevated at 3.4 nanograms per milliliter.
The patient was started on aspirin 325 milligrams, clopidogrel 600 milligrams, and intravenous heparin.
Emergency percutaneous coronary intervention was performed with placement of a drug-eluting stent in the right coronary artery.
Post-procedure the patient is hemodynamically stable with resolution of chest pain.
Echocardiogram demonstrates inferior wall hypokinesis with an ejection fraction of 45 percent.
The patient is admitted to the coronary care unit for monitoring and further management.
Discharge medications include aspirin, clopidogrel, metoprolol, lisinopril, and atorvastatin.

History of present illness reveals a 67-year-old male with a history of hypertension, hyperlipidemia, and type 2 diabetes mellitus.
The patient reports progressive dyspnea on exertion over the past three weeks, orthopnea, and bilateral ankle edema.
Physical examination reveals jugular venous distension, bibasilar crackles, and 2-plus pitting edema of the lower extremities.
Chest radiograph demonstrates cardiomegaly with pulmonary vascular congestion and bilateral pleural effusions.
Brain natriuretic peptide level is 1240 picograms per milliliter, significantly elevated.
The diagnosis of acute decompensated heart failure is established.
Intravenous furosemide is initiated at 80 milligrams twice daily with close monitoring of fluid balance and electrolytes.
The patient achieves a net negative fluid balance of 2.5 liters over 48 hours with improvement in symptoms.
Cardiology consultation is obtained and the patient is scheduled for outpatient cardiac rehabilitation.

Radiology report indicates a 2.3 centimeter spiculated pulmonary nodule in the right upper lobe with associated mediastinal lymphadenopathy.
CT-guided biopsy is performed under local anesthesia with conscious sedation.
Pathology confirms non-small cell lung carcinoma, adenocarcinoma subtype, with EGFR exon 19 deletion mutation.
Positron emission tomography scan reveals no evidence of distant metastatic disease, staging the tumor as T2aN2M0, stage IIIA.
The multidisciplinary tumor board recommends concurrent chemoradiation therapy followed by consolidation immunotherapy.
Treatment is initiated with cisplatin plus pemetrexed chemotherapy and concurrent thoracic radiation therapy to 60 Gray in 30 fractions.
The patient tolerates treatment with manageable toxicity including grade 2 fatigue and grade 1 nausea.
Consolidation durvalumab immunotherapy is commenced 4 weeks following completion of chemoradiation.
Follow-up imaging at 3 months demonstrates significant reduction in primary tumor size and lymphadenopathy.

The patient is a 34-year-old female presenting with fever, severe headache, photophobia, and neck stiffness.
Neurological examination reveals positive Kernig and Brudzinski signs.
Lumbar puncture is performed with opening pressure of 32 centimeters of water.
Cerebrospinal fluid analysis shows turbid appearance, white blood cell count of 1800 cells per microliter with 90 percent neutrophils,
glucose of 28 milligrams per deciliter, and protein of 320 milligrams per deciliter.
Gram stain demonstrates gram-positive diplococci. Culture confirms Streptococcus pneumoniae.
Bacterial meningitis is diagnosed and intravenous ceftriaxone 2 grams every 12 hours and dexamethasone are started emergently.
Blood cultures also grow Streptococcus pneumoniae sensitive to cephalosporins.
The patient improves clinically over 7 days of antibiotic therapy with resolution of fever and meningismus.
Audiological testing at discharge reveals mild sensorineural hearing loss in the right ear.

Assessment and plan for a 58-year-old female with poorly controlled type 2 diabetes mellitus.
Hemoglobin A1c is 10.2 percent indicating suboptimal glycemic control over the preceding 3 months.
Fasting plasma glucose is 248 milligrams per deciliter. Urine albumin to creatinine ratio is 145 milligrams per gram.
Ophthalmology evaluation reveals background diabetic retinopathy without macular edema.
Metformin dose is increased to 1000 milligrams twice daily. Semaglutide subcutaneous injection is added.
Comprehensive diabetes education and medical nutrition therapy are reinforced.
Blood pressure is 138 over 84 mmHg; lisinopril dose is titrated upward for renoprotection.
Aspirin 81 milligrams daily is continued for cardiovascular risk reduction.
Referral is placed for nephrology evaluation given progressive microalbuminuria.
The patient is scheduled for repeat laboratory evaluation and clinic visit in 3 months.
""".strip()

# Build character vocabulary
chars   = sorted(set(MEDICAL_TEXT))
vocab   = {ch: i for i, ch in enumerate(chars)}
inv_vocab = {i: ch for ch, i in vocab.items()}
VOCAB_SIZE = len(vocab)

print(f"Corpus length  : {len(MEDICAL_TEXT):,} characters")
print(f"Vocabulary size: {VOCAB_SIZE} unique characters")
print(f"Sample chars   : {chars[:20]}")

In [ ]:
# ── Given: Sequence dataset for character-level modelling ────────────────────────────
class CharDataset(Dataset):
    def __init__(self, text, seq_len, vocab):
        self.seq_len = seq_len
        self.vocab   = vocab
        self.data    = [vocab[c] for c in text if c in vocab]

    def __len__(self):
        return max(0, len(self.data) - self.seq_len)

    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx:idx + self.seq_len],          dtype=torch.long)
        y = torch.tensor(self.data[idx + 1:idx + self.seq_len + 1],  dtype=torch.long)
        return x, y

char_dataset  = CharDataset(MEDICAL_TEXT, Config.RNN_SEQ_LEN, vocab)
char_loader   = DataLoader(char_dataset, batch_size=Config.RNN_BATCH_SIZE, shuffle=True, num_workers=0)
print(f"Training sequences: {len(char_dataset):,}")
print(f"Batches per epoch : {len(char_loader)}")

---
### 2a. Character-Level LSTM Model

In [ ]:
class CharLSTM(nn.Module):
    """
    Character-level LSTM language model.

    Architecture:
      Embedding(vocab_size, embed_dim=128)
      → LSTM(embed_dim, hidden_size, num_layers, dropout, batch_first=True)
      → Linear(hidden_size, vocab_size)

    forward(x, hidden=None) returns (logits, new_hidden)
      x      : [B, T] token indices
      logits : [B, T, vocab_size]
    """
    def __init__(self, vocab_size, hidden_size, num_layers, dropout):
        super().__init__()
        # ============================================================
        # TODO: Define self.embedding, self.lstm, self.fc
        # ============================================================
        self.embedding = None   # TODO: nn.Embedding(vocab_size, 128)
        self.lstm      = None   # TODO: nn.LSTM(...)
        self.fc        = None   # TODO: nn.Linear(hidden_size, vocab_size)
        # ============================================================

    def forward(self, x, hidden=None):
        # ============================================================
        # TODO: Pass x through embedding → lstm → fc
        #       Return (logits, hidden)
        # ============================================================
        pass  # Remove after completing TODO
        # ============================================================

    def init_hidden(self, batch_size, device):
        """Initialise hidden and cell states to zeros."""
        h0 = torch.zeros(Config.RNN_NUM_LAYERS, batch_size, Config.RNN_HIDDEN_SIZE).to(device)
        c0 = torch.zeros(Config.RNN_NUM_LAYERS, batch_size, Config.RNN_HIDDEN_SIZE).to(device)
        return (h0, c0)

char_model = CharLSTM(VOCAB_SIZE, Config.RNN_HIDDEN_SIZE, Config.RNN_NUM_LAYERS, Config.RNN_DROPOUT).to(device)
rnn_optimizer = optim.Adam(char_model.parameters(), lr=Config.RNN_LR)
rnn_criterion = nn.CrossEntropyLoss()
print(f"CharLSTM parameters: {sum(p.numel() for p in char_model.parameters()):,}")

---
### 2b. Training Loop

In [ ]:
def train_char_rnn(model, loader, criterion, optimizer, epochs, device):
    """Train the character-level LSTM and return per-epoch loss history."""
    history = []
    model.train()

    print(f"\n{'='*50}\nTraining Character-Level LSTM\n{'='*50}")
    for epoch in range(epochs):
        # ============================================================
        # TODO: Iterate over loader, compute loss, backpropagate.
        #       - Detach hidden state between batches to avoid
        #         backpropagating through the full data sequence.
        #       - Clip gradients to max_norm=5.0 with
        #         nn.utils.clip_grad_norm_ to prevent exploding gradients.
        #       - Track total loss and print epoch average.
        # ============================================================
        epoch_loss = 0.0
        num_batches = 0

        for x_batch, y_batch in tqdm(loader, desc=f'Epoch {epoch+1}', leave=False):
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            # TODO: init hidden, forward, compute loss, backward, clip, step
            pass  # Remove after completing TODO

        avg_loss = epoch_loss / max(num_batches, 1)
        history.append(avg_loss)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Perplexity: {np.exp(avg_loss):.1f}")
        # ============================================================

    return history

rnn_history = train_char_rnn(char_model, char_loader, rnn_criterion, rnn_optimizer, Config.RNN_EPOCHS, device)

In [ ]:
# ── Given: Plot RNN training loss ──────────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(rnn_history, marker='o')
plt.title('Character-Level LSTM — Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.tight_layout()
plt.show()

---
### 2c. Text Generation

In [ ]:
def generate_medical_text(model, seed_str, length, vocab, inv_vocab, device, temperature=0.8):
    """
    Generate `length` characters of medical text starting from `seed_str`.

    Args:
        model       : trained CharLSTM
        seed_str    : starting string (must contain chars in vocab)
        length      : number of characters to generate
        vocab       : char → index mapping
        inv_vocab   : index → char mapping
        device      : torch device
        temperature : sampling temperature (lower = more deterministic)

    Returns:
        generated (str) : seed_str + newly generated characters
    """
    model.eval()
    # ============================================================
    # TODO: Encode seed_str as token indices.
    #       Warm up the model by feeding the seed tokens one by one.
    #       Then generate `length` new characters by:
    #         1. Forward pass to get logits for the last position
    #         2. Divide logits by temperature
    #         3. Sample from softmax distribution with torch.multinomial
    #         4. Append sampled character to generated string
    # ============================================================
    generated = seed_str
    # ... your implementation here ...
    pass  # Remove after completing TODO
    # ============================================================
    return generated

# Test generation
seed = "The patient presents with"
generated_text = generate_medical_text(
    char_model, seed, length=300, vocab=vocab,
    inv_vocab=inv_vocab, device=device, temperature=0.8
)
print("Generated text:\n")
print(generated_text)

---
## Problem 3: RNN with Attention for Medical Text Classification

In this problem you will implement an **RNN encoder with an additive attention mechanism**  
to classify clinical notes by diagnostic category.  
After training, you will visualise the attention weights to interpret what the model focuses on.

In [ ]:
# ── Given: Synthetic clinical-note classification dataset ────────────────────────────
CLINICAL_NOTES = [
    # (note_text, label)   0=Cardiology  1=Pulmonology  2=Neurology  3=Infectious Disease
    ("patient presents with chest pain and shortness of breath, elevated troponin, ST changes on ECG", 0),
    ("acute myocardial infarction with cardiogenic shock, requires urgent revascularization", 0),
    ("hypertensive urgency with blood pressure 200 over 120, started on labetalol drip", 0),
    ("atrial fibrillation with rapid ventricular response, rate controlled with diltiazem", 0),
    ("heart failure exacerbation with bilateral pulmonary edema, BNP elevated", 0),
    ("unstable angina, catheterization shows three vessel disease, referred for bypass", 0),
    ("ventricular tachycardia storm, defibrillated three times, amiodarone infusion started", 0),
    ("pericarditis with friction rub and diffuse ST elevation, started on colchicine", 0),
    ("dyspnea on exertion, reduced diffusing capacity, diagnosis of pulmonary fibrosis", 1),
    ("chronic obstructive pulmonary disease exacerbation, hypercapnia, non-invasive ventilation", 1),
    ("pulmonary embolism with right heart strain on echo, anticoagulation initiated", 1),
    ("asthma exacerbation unresponsive to bronchodilators, admitted to ICU", 1),
    ("community acquired pneumonia, lobar consolidation, started on ceftriaxone", 1),
    ("lung adenocarcinoma with pleural effusion, thoracentesis performed", 1),
    ("obstructive sleep apnea with hypoxia, initiated on CPAP therapy", 1),
    ("sarcoidosis with bilateral hilar adenopathy, pulmonary function tests show restriction", 1),
    ("ischemic stroke with left hemiplegia, tPA administered within treatment window", 2),
    ("seizure disorder, status epilepticus, lorazepam and levetiracetam given", 2),
    ("bacterial meningitis, CSF pleocytosis, neutrophilic, started on ceftriaxone", 2),
    ("Parkinson disease with motor fluctuations, carbidopa leveodopa adjusted", 2),
    ("multiple sclerosis relapse, optic neuritis, high dose steroids administered", 2),
    ("subarachnoid hemorrhage from ruptured aneurysm, neurosurgery consulted", 2),
    ("Guillain-Barre syndrome with ascending paralysis, IVIG therapy initiated", 2),
    ("migraine with aura, refractory to triptans, admitted for dihydroergotamine", 2),
    ("septic shock from urinary source, blood cultures positive for E. coli", 3),
    ("HIV with CD4 count 45, Pneumocystis pneumonia prophylaxis added", 3),
    ("Clostridioides difficile colitis, oral vancomycin initiated, contact precautions", 3),
    ("endocarditis with vegetations on mitral valve, six week antibiotic course planned", 3),
    ("tuberculosis with cavitary lesion, four drug regimen initiated, respiratory isolation", 3),
    ("neutropenic fever in oncology patient, broad spectrum antibiotics started empirically", 3),
    ("COVID-19 pneumonia requiring supplemental oxygen, remdesivir and dexamethasone given", 3),
    ("osteomyelitis of the tibia, debridement and six weeks of intravenous antibiotics", 3),
]

CLASS_NAMES = ['Cardiology', 'Pulmonology', 'Neurology', 'Infectious Disease']
NUM_CLASSES  = len(CLASS_NAMES)

print(f"Clinical notes: {len(CLINICAL_NOTES)}")
print(f"Classes: {CLASS_NAMES}")
for i, name in enumerate(CLASS_NAMES):
    count = sum(1 for _, l in CLINICAL_NOTES if l == i)
    print(f"  {i} {name}: {count} samples")

In [ ]:
# ── Given: Tokenisation and DataLoader ───────────────────────────────────────────────
def build_word_vocab(notes):
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for text, _ in notes:
        for word in text.lower().split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

word_vocab    = build_word_vocab(CLINICAL_NOTES)
WORD_VOCAB_SIZE = len(word_vocab)
print(f"Word vocabulary size: {WORD_VOCAB_SIZE}")

class ClinicalNoteDataset(Dataset):
    def __init__(self, notes, vocab, max_len):
        self.samples = []
        for text, label in notes:
            tokens = [vocab.get(w, 1) for w in text.lower().split()]
            # Pad or truncate to max_len
            if len(tokens) < max_len:
                tokens += [0] * (max_len - len(tokens))
            else:
                tokens = tokens[:max_len]
            self.samples.append((torch.tensor(tokens, dtype=torch.long), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return x, torch.tensor(y, dtype=torch.long)

# 80/20 split
random.shuffle(CLINICAL_NOTES)
split = int(0.8 * len(CLINICAL_NOTES))
train_notes, val_notes = CLINICAL_NOTES[:split], CLINICAL_NOTES[split:]

cls_train_ds = ClinicalNoteDataset(train_notes, word_vocab, Config.CLS_MAX_LEN)
cls_val_ds   = ClinicalNoteDataset(val_notes,   word_vocab, Config.CLS_MAX_LEN)

cls_train_loader = DataLoader(cls_train_ds, batch_size=Config.CLS_BATCH_SIZE, shuffle=True,  num_workers=0)
cls_val_loader   = DataLoader(cls_val_ds,   batch_size=Config.CLS_BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Train: {len(cls_train_ds)} | Val: {len(cls_val_ds)}")

---
### 3a. Attention Mechanism

In [ ]:
class AdditiveAttention(nn.Module):
    """
    Bahdanau-style additive attention.

    Given:
      hidden  : final RNN hidden state       [B, hidden_size]
      outputs : all RNN step outputs         [B, T, hidden_size]

    Returns:
      context : weighted sum of outputs      [B, hidden_size]
      weights : attention distribution       [B, T]

    Architecture:
      score(h_t, s) = v^T · tanh(W_h · h_t + W_s · s)
      alpha          = softmax(scores)
      context        = sum_t alpha_t * h_t
    """
    def __init__(self, hidden_size):
        super().__init__()
        # ============================================================
        # TODO: Define W_h, W_s (both Linear(hidden_size, hidden_size)),
        #       and v (Linear(hidden_size, 1, bias=False))
        # ============================================================
        self.W_h = None  # TODO
        self.W_s = None  # TODO
        self.v   = None  # TODO
        # ============================================================

    def forward(self, hidden, outputs):
        """
        hidden : [B, hidden_size]
        outputs: [B, T, hidden_size]
        """
        # ============================================================
        # TODO: Compute additive attention scores, apply softmax,
        #       compute context as weighted sum of outputs.
        #       Return (context, weights).
        # ============================================================
        pass  # Remove after completing TODO
        # ============================================================

---
### 3b. RNN-with-Attention Classifier

In [ ]:
class RNNAttentionClassifier(nn.Module):
    """
    Bi-GRU encoder + additive attention + linear classifier.

    Architecture:
      Embedding(vocab_size, embed_dim)
      → GRU(embed_dim, hidden_size, bidirectional=True, batch_first=True)
      → AdditiveAttention(hidden_size * 2)
      → Linear(hidden_size * 2, num_classes)
    """
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes, dropout=0.3):
        super().__init__()
        # ============================================================
        # TODO: Define self.embedding, self.gru (bidirectional),
        #       self.attention (AdditiveAttention), self.dropout, self.fc
        #       Note: bidirectional GRU outputs hidden_size*2 features
        # ============================================================
        self.embedding = None   # TODO
        self.gru       = None   # TODO
        self.attention = None   # TODO: AdditiveAttention(hidden_size * 2)
        self.dropout   = None   # TODO
        self.fc        = None   # TODO
        # ============================================================

    def forward(self, x):
        """
        x : [B, T] token indices
        Returns (logits [B, num_classes], attn_weights [B, T])
        """
        # ============================================================
        # TODO: embedding → gru → attention → dropout → fc
        #       For bidirectional GRU, concatenate the final forward
        #       and backward hidden states to form the query for attention.
        # ============================================================
        pass  # Remove after completing TODO
        # ============================================================

cls_model     = RNNAttentionClassifier(WORD_VOCAB_SIZE, Config.CLS_EMBED_DIM,
                                       Config.CLS_HIDDEN_SIZE, NUM_CLASSES).to(device)
cls_criterion = nn.CrossEntropyLoss()
cls_optimizer = optim.Adam(cls_model.parameters(), lr=Config.CLS_LR)
print(f"RNNAttentionClassifier parameters: {sum(p.numel() for p in cls_model.parameters()):,}")

---
### 3c. Training and Evaluation

In [ ]:
def train_cls_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, n = 0.0, 0, 0

    for x_batch, y_batch in tqdm(loader, desc='Cls Train', leave=False):
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        # ============================================================
        # TODO: Forward pass (remember model returns logits AND weights),
        #       compute loss, backward, clip gradients (max_norm=1.0), step.
        # ============================================================
        optimizer.zero_grad()
        logits, _ = None, None   # TODO: forward
        loss       = None        # TODO: criterion
        # TODO: backward, clip, step
        # ============================================================

        total_loss += loss.item() * x_batch.size(0)
        preds       = logits.argmax(dim=1)
        correct    += (preds == y_batch).sum().item()
        n          += x_batch.size(0)

    return total_loss / n, 100. * correct / n


def eval_cls_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    all_preds, all_labels  = [], []

    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            logits, _ = model(x_batch)
            loss = criterion(logits, y_batch)
            total_loss += loss.item() * x_batch.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            n += x_batch.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    return total_loss / n, 100. * correct / n, all_preds, all_labels


def train_classifier(model, train_loader, val_loader, criterion, optimizer, epochs, device):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    print(f"\n{'='*50}\nTraining RNN-Attention Classifier\n{'='*50}")
    for epoch in range(epochs):
        # ============================================================
        # TODO: Call train_cls_epoch and eval_cls_epoch,
        #       append to history, print epoch summary.
        # ============================================================
        pass  # Remove after completing TODO
        # ============================================================

    return model, history

cls_model, cls_history = train_classifier(
    cls_model, cls_train_loader, cls_val_loader,
    cls_criterion, cls_optimizer, Config.CLS_EPOCHS, device
)

In [ ]:
# ── Given: Plot classification learning curves ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(cls_history['train_loss'], label='Train')
axes[0].plot(cls_history['val_loss'],   label='Val')
axes[0].set_title('Classification Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(cls_history['train_acc'], label='Train')
axes[1].plot(cls_history['val_acc'],   label='Val')
axes[1].set_title('Classification Accuracy (%)')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

# Final evaluation
_, val_acc, val_preds, val_labels = eval_cls_epoch(cls_model, cls_val_loader, cls_criterion, device)
print(f"\nValidation Accuracy: {val_acc:.1f}%\n")
print(classification_report(val_labels, val_preds, target_names=CLASS_NAMES, zero_division=0))

---
### 3d. Attention Weight Visualization

Visualise which words the model attends to when classifying a clinical note.

In [ ]:
def visualise_attention(model, note_text, vocab, class_names, device, max_len=50):
    """
    Run a single clinical note through the model and plot the attention weights
    as a bar chart with word labels on the x-axis.
    """
    # ============================================================
    # TODO: Tokenise note_text using vocab (same as ClinicalNoteDataset),
    #       run through model to get (logits, attn_weights),
    #       determine the predicted class,
    #       and plot a bar chart of attention weight per token word.
    # ============================================================
    model.eval()
    words  = note_text.lower().split()[:max_len]
    tokens = [vocab.get(w, 1) for w in words]
    if len(tokens) < max_len:
        tokens += [0] * (max_len - len(tokens))
    x = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        logits, attn_weights = None, None  # TODO: model(x)

    # TODO: Extract weights for the non-padded word positions,
    #       plot horizontal bar chart, colour bars by weight magnitude.
    pass  # Remove after completing TODO
    # ============================================================

# Visualise attention for one note from each class
for note_text, label in CLINICAL_NOTES[:4]:
    visualise_attention(cls_model, note_text, word_vocab, CLASS_NAMES, device)

---
## Analysis Questions (20 points)

Answer each question in the markdown cell below it. Write 3–5 sentences per answer.

### Question 1 (5 points)

**Attention gates vs. standard skip connections in segmentation**

In the Attention U-Net (Problem 1), attention gates filter the skip-connection features before they are concatenated with the decoder.  
Explain how this differs from a standard U-Net skip connection.  
In what scenarios — e.g., small lesions, noisy backgrounds, class imbalance — would attention gates provide the greatest benefit, and why?

**Your Answer:**

*[Write your answer here]*

### Question 2 (5 points)

**Vanishing gradients and LSTM**

Vanilla RNNs suffer from the vanishing gradient problem when trained on long sequences.  
Explain the mathematical reason this happens during backpropagation through time.  
Then describe how the LSTM architecture (specifically the forget gate, input gate, and cell state) addresses this problem.

**Your Answer:**

*[Write your answer here]*

### Question 3 (5 points)

**Interpreting attention weights (Problem 3)**

Look at the attention visualizations you produced for at least two clinical notes from different specialties.  
Which words or phrases received the highest attention weights?  
Do the high-attention words align with what a clinician would consider the most diagnostically relevant terms?  
Discuss one way that attention weight analysis could assist clinicians or improve model trustworthiness.

**Your Answer:**

*[Write your answer here]*

### Question 4 (5 points)

**Comparing the three attention mechanisms**

This homework used three different forms of attention:  
(a) attention gates in the segmentation U-Net,  
(b) the LSTM hidden state implicitly weighting which characters matter for next-token prediction,  
(c) explicit additive attention in the RNN classifier.  
Compare these three mechanisms: when is each one most beneficial, and what are the trade-offs in terms of interpretability, computational cost, and task suitability?

**Your Answer:**

*[Write your answer here]*

---
## Submission Instructions

1. Run all cells top-to-bottom and confirm there are no errors
2. Rename this notebook: **`FirstName_LastName_HW02.ipynb`**
3. Submit via the course portal by the posted deadline

**Your submission must show:**
- All TODOs completed with working code
- Training output printed for all three problems
- All plots rendered (segmentation curves, RNN loss, classification curves, attention charts)
- Analysis questions answered in complete sentences

**Example filename:** `Jane_Smith_HW02.ipynb`